# 05 · Covariates: the mode where Chronos-2 pulls ahead

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/chronos2-in-practice/blob/main/notebooks/05_covariates.ipynb)

Companion notebook for *Chronos-2 in Practice (3/5)*.

**What you will learn**

- The difference between a past covariate and a known-future one, which in `predict_df` is
  decided by nothing more than whether a column appears in `future_df`.
- A replication of fev-bench's EPF-DE task — German day-ahead price with the grid operator's
  load and renewable forecasts — checked against the published cutoffs rather than merely
  resembling it.
- How much of the gain comes from *having* the covariate series versus from *knowing its
  future*: three arms on the same task separate the two.
- The same design on Spanish prices as a ladder: nothing, calendar, published day-ahead
  forecasts, past generation.
- How much a leak flatters you, measured on purpose: the same ladder with observed values
  passed off as known-future ones.

Runtime: ~25 min on a GPU (FAST_MODE: ~3 min), plus a download of the EPF-DE dataset.

In [ ]:
# On Google Colab, clone the companion repo and install it (Colab's own PyTorch is kept).
# Locally, `uv sync` already installed everything and this cell does nothing.
import sys

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/YOUR_GITHUB_USER/chronos2-in-practice.git
    %cd chronos2-in-practice
    %pip install -q .
    import importlib

    importlib.invalidate_caches()

In [ ]:
import time
from dataclasses import replace

import holidays
import pandas as pd
from huggingface_hub.utils import logging as hub_logging
from IPython.display import Image, display
from transformers.utils import logging as transformers_logging

from c2p.config import get_device, get_paths, is_fast_mode, set_seed
from c2p.data import (
    CALENDAR_COLUMNS,
    add_calendar_features,
    spain_energy_frame,
    validate_and_prepare,
)
from c2p.eval import backtest, evaluate, skill_table
from c2p.experiments import FREQ, PRICE, TSO_COVARIATES, day_ahead_config
from c2p.fev_tasks import (
    EPF_DE,
    assert_cutoffs_match,
    load_fev_task,
    task_backtest_config,
    task_series_frame,
)
from c2p.models import Chronos2Forecaster, load_chronos2
from c2p.plots.evaluation import plot_skill_scores
from c2p.plotting import save_figure, set_style
from c2p.results import load_results, run_metadata, save_results
from c2p.schema import ID_COL, TIMESTAMP_COL

FAST_MODE = is_fast_mode(default=False)
PATHS = get_paths(fast_mode=FAST_MODE)
set_seed()
DEVICE = get_device()
set_style()
hub_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

for key, value in run_metadata(device=DEVICE, fast_mode=FAST_MODE).items():
    print(f"{key:>20}: {value}")

## 1. One rule, two kinds of covariate

Chronos-2 takes covariates as *series in the same group as the target*. Which kind a column is
depends on one thing only:

- a column of `df` that is **not** in `future_df` is a **past covariate**: the model sees its
  history and nothing after the cutoff;
- a column present in **both** is a **known-future covariate**: the model also sees its values
  across the horizon.

There is no third option and no flag. In this repository the backtest harness makes the choice
explicit instead: only columns listed in `BacktestConfig.known_covariates` are put into the
future frame, and a guard checks that nothing else crossed the cutoff. Everything else in the
frame is a past covariate by construction.

That is also where the most expensive mistake in applied forecasting lives. A "known-future"
covariate must be knowable *at forecast time*: a published day-ahead forecast is, the value that
was eventually observed is not. Section 4 measures what the second one does to your results.

In [ ]:
pipeline = load_chronos2(device=DEVICE)
chronos = Chronos2Forecaster(pipeline)

## 2. Replicating EPF-DE from fev-bench

The Chronos-2 paper reports its largest margins on covariate tasks, and EPF-DE is the clearest
of them: the German day-ahead electricity price, forecast 24 hours ahead with two covariates the
market genuinely knows in advance — Amprion's load forecast and a combined PV + wind forecast.

"Replicating" it means using the published task, not one that looks like it. `c2p.fev_tasks`
loads the task from the fev-bench task list and restates it as a `BacktestConfig`, then
`assert_cutoffs_match` checks that our 20 windows land on fev's 20 published cutoffs. If the
translation were off by one step, that assertion fails and the replication stops here.

In [ ]:
task = load_fev_task(EPF_DE)
epf = task_series_frame(task)
EPF_CONFIG = task_backtest_config(task)
print(f"task:             {task.task_name}")
print(f"target:           {task.target}")
print(f"known covariates: {task.known_dynamic_columns}")
print(
    f"horizon:          {task.horizon} h, {task.num_windows} windows, "
    f"step {task.window_step_size} h, seasonality {task.seasonality}"
)
print(
    f"data:             {len(epf):,} rows, {epf[TIMESTAMP_COL].min()} to {epf[TIMESTAMP_COL].max()}"
)

assert_cutoffs_match(task, epf, EPF_CONFIG)
print("\nOur windows sit on fev's published cutoffs.")

Three arms separate two effects that are easy to confuse. `univariate` drops the covariate
columns entirely. `past_only` keeps them in the frame but does not declare them known, so the
model sees their history and stops at the cutoff. `known_future` is the published task: the same
columns, with their horizon values supplied.

If most of the gain appeared in `past_only`, covariates would simply be extra context. If it
appears in `known_future`, what the model is using is the future it was handed.

In [ ]:
EPF_ARMS = {
    "univariate": (epf[[ID_COL, TIMESTAMP_COL, task.target]], ()),
    "past_only": (epf, ()),
    "known_future": (epf, tuple(task.known_dynamic_columns)),
}
if FAST_MODE:
    EPF_CONFIG = replace(EPF_CONFIG, n_windows=5)

frames = []
for name, (frame, known) in EPF_ARMS.items():
    config = replace(EPF_CONFIG, known_covariates=known)
    model = Chronos2Forecaster(pipeline, name=name)
    metrics = evaluate(backtest(model, frame, config), frame, config)
    frames.append(metrics)
    print(f"  {name:<14} WQL {metrics['WQL'].mean():.4f}  SQL {metrics['SQL'].mean():.4f}")
epf_metrics = pd.concat(frames, ignore_index=True)
print("Saved", save_results(epf_metrics, "05_epf_de_metrics", PATHS).name)

In [ ]:
epf_metrics = load_results("05_epf_de_metrics", PATHS)
epf_skill = skill_table(epf_metrics, baseline="univariate", metric="WQL")
print("Saved", save_results(epf_skill, "05_epf_de_skill", PATHS).name)

known = epf_skill.set_index("model").loc["known_future"]
past = epf_skill.set_index("model").loc["past_only"]
title = (
    f"EPF-DE: knowing the covariates' future cuts quantile loss by {known['skill']:.0%}; "
    f"their history alone, {past['skill']:+.0%}"
)
ax = plot_skill_scores(
    epf_skill,
    highlight="known_future",
    title=title,
    xlabel="WQL improvement over the univariate task (%)",
)
(figure_path,) = save_figure(ax.figure, "05_epf_de_covariates", PATHS.figures)
display(Image(filename=figure_path, width=800))

## 3. The same design on Spanish prices

The Spanish dataset carries the equivalent covariates: the grid operator publishes day-ahead
forecasts of load, solar and onshore wind the day before, so for a horizon that ends within the
next day they are legitimately known at forecast time. The target is the hourly spot price.

The ladder adds one kind of information at a time, so each rung is attributable:

| Rung | Added |
| --- | --- |
| `none` | nothing: the price history alone |
| `+calendar` | hour, day of week, month, weekend and Spanish holidays, as known-future |
| `+TSO forecasts` | the operator's day-ahead load, solar and wind forecasts, as known-future |
| `+past generation` | actual generation by source, as **past-only** covariates |
| `LEAK: observed as known` | the same quantities, but **observed** values handed over as if known |
| `LEAK: the target itself` | a copy of the price, declared known-future — the unmistakable case |

Note what is *not* compared: the price error in EUR/MWh against the German task above. Different
markets, different price levels, different years. Only the relative improvement travels.

In [ ]:
COLUMNS = [ID_COL, TIMESTAMP_COL, PRICE, *TSO_COVARIATES]
LEAK_COLUMNS = ("load_actual", "gen_solar", "gen_wind_onshore")
PAST_COLUMNS = ("gen_solar", "gen_wind_onshore", "gen_nuclear", "gen_fossil_gas")
raw = spain_energy_frame()
price, price_report = validate_and_prepare(
    raw[[*COLUMNS, *dict.fromkeys([*LEAK_COLUMNS, *PAST_COLUMNS])]], freq=FREQ, target=PRICE
)
price = add_calendar_features(price)
price["price_leak"] = price[PRICE]  # a copy of the target, for the leak demonstration below
CALENDAR = tuple(CALENDAR_COLUMNS)
print(price_report)
print(f"\nCalendar covariates: {CALENDAR}")
print(f"TSO day-ahead forecasts: {TSO_COVARIATES}")

In [ ]:
LADDER = {
    "none": {"columns": (), "known": ()},
    "+calendar": {"columns": CALENDAR, "known": CALENDAR},
    "+TSO forecasts": {
        "columns": (*CALENDAR, *TSO_COVARIATES),
        "known": (*CALENDAR, *TSO_COVARIATES),
    },
    "+past generation": {
        "columns": (*CALENDAR, *TSO_COVARIATES, *PAST_COLUMNS),
        "known": (*CALENDAR, *TSO_COVARIATES),
    },
    "LEAK: observed as known": {
        "columns": (*CALENDAR, *LEAK_COLUMNS),
        "known": (*CALENDAR, *LEAK_COLUMNS),
    },
    "LEAK: the target itself": {
        "columns": (*CALENDAR, "price_leak"),
        "known": (*CALENDAR, "price_leak"),
    },
}

start = time.perf_counter()
frames = []
for name, spec in LADDER.items():
    frame = price[[ID_COL, TIMESTAMP_COL, PRICE, *spec["columns"]]]
    config = day_ahead_config(
        FAST_MODE, target=PRICE, known_covariates=spec["known"], max_context=2048
    )
    model = Chronos2Forecaster(pipeline, name=name)
    metrics = evaluate(backtest(model, frame, config), frame, config)
    frames.append(metrics)
    print(
        f"  {name:<24} WQL {metrics['WQL'].mean():.4f}  cov80 {metrics['coverage_80'].mean():.1%}"
    )
ladder = pd.concat(frames, ignore_index=True)
print(
    f"{len(LADDER)} rungs x {config.n_windows} windows in "
    f"{(time.perf_counter() - start) / 60:.1f} min"
)
print("Saved", save_results(ladder, "05_price_ladder_metrics", PATHS).name)

In [ ]:
ladder = load_results("05_price_ladder_metrics", PATHS)
ladder_skill = skill_table(ladder, baseline="none", metric="WQL")
print("Saved", save_results(ladder_skill, "05_price_ladder_skill", PATHS).name)
ladder_skill.round(4)

In [ ]:
ladder_skill = load_results("05_price_ladder_skill", PATHS)
honest = ladder_skill[~ladder_skill["model"].str.startswith("LEAK")]
best = honest.iloc[0]
title = (
    f"Spanish prices: the operator's published forecasts are worth "
    f"{best['skill']:.0%} of the quantile loss"
)
ax = plot_skill_scores(
    ladder_skill,
    highlight=str(best["model"]),
    title=title,
    xlabel="WQL improvement over no covariates (%)",
)
(figure_path,) = save_figure(ax.figure, "05_covariate_ladder", PATHS.figures)
display(Image(filename=figure_path, width=800))

## 4. The trap, and what it actually looks like

The mistake every covariate pipeline can make is handing the model a column that is only known
*after* the fact. Nothing in the API objects — the leak is in the data, not in the call — so the
only defence is knowing what a leak looks like in the results.

Two arms probe it, and they do not agree, which is the interesting part:

- `LEAK: observed as known` replaces the operator's published forecasts with the values that
  were actually observed. On this market that changes very little, and the reason is economic
  rather than statistical: the day-ahead price is *cleared against the day-ahead forecasts*. The
  published forecast is not a noisy proxy for the outcome — it is close to the input the market
  actually used. A leak is not automatically an advantage.
- `LEAK: the target itself` hands over a copy of the price as a known-future covariate. That is
  what a leak looks like when it bites, and the number below is the signature to watch for.

The practical rule is therefore not "leaks inflate results by X%" but: an implausibly large gain
from a covariate is a reason to audit the column, not to publish it. And when a covariate exists
in both a forecast and an observed version, prefer the forecast — not only because it is honest,
but because on this dataset it is also the more informative one.

In [ ]:
ladder_skill = load_results("05_price_ladder_skill", PATHS).set_index("model")
honest = ladder_skill.loc["+TSO forecasts", "skill"]
observed = ladder_skill.loc["LEAK: observed as known", "skill"]
target_leak = ladder_skill.loc["LEAK: the target itself", "skill"]
print(
    f"Published day-ahead forecasts, known-future: {honest:+7.1%}\n"
    f"Observed values passed off as known:         {observed:+7.1%}  "
    f"({100 * (observed - honest):+.1f} points vs the honest arm)\n"
    f"The target itself, passed off as known:      {target_leak:+7.1%}  "
    f"<- the signature of a real leak"
)

## 5. Holidays, measured where they happen

The calendar rung above bundles five features, and four of them are *periodic*: hour, weekday,
month, weekend. The model can read all four off 2,048 hours of price history, so a null result
for the bundle says little about the fifth, `is_holiday`, which is the one thing positions cannot
encode. 2018 has ten national holidays, so ten of the 365 windows forecast a holiday — too few
to move the yearly skill score whatever happens on them.

Two things settle it. A rung with *only* the holiday flag, so nothing periodic dilutes it; and the
skill of both calendar rungs split by whether the window's horizon is a holiday. The horizon of a
22:00 UTC cutoff is the next local day, so a window is a holiday window when that day is in the
national calendar.


In [ ]:
HOLIDAY_ARM = "+holiday flag only"
holiday_frame = price[[ID_COL, TIMESTAMP_COL, PRICE, "is_holiday"]]
holiday_config = day_ahead_config(
    FAST_MODE, target=PRICE, known_covariates=("is_holiday",), max_context=2048
)
holiday_metrics = evaluate(
    backtest(Chronos2Forecaster(pipeline, name=HOLIDAY_ARM), holiday_frame, holiday_config),
    holiday_frame,
    holiday_config,
)
ladder = load_results("05_price_ladder_metrics", PATHS)
arms = pd.concat(
    [ladder[ladder["model"].isin(["none", "+calendar"])], holiday_metrics], ignore_index=True
)

# A window forecasts a holiday when its horizon's local date is in the national calendar.
horizon_day = (
    (pd.to_datetime(arms["cutoff"]) + pd.Timedelta(hours=2))
    .dt.tz_localize("UTC")
    .dt.tz_convert("Europe/Madrid")
    .dt.normalize()
    .dt.tz_localize(None)
)
holiday_dates = pd.to_datetime(list(holidays.country_holidays("ES", years=[2018])))
arms["holiday_window"] = horizon_day.isin(holiday_dates)
n_holiday = int(arms.loc[arms["model"] == "none", "holiday_window"].sum())
print(f"{n_holiday} holiday windows among {holiday_config.n_windows}")

subsets = {
    "all windows": arms,
    "holiday windows": arms[arms["holiday_window"]],
    "ordinary windows": arms[~arms["holiday_window"]],
}
holiday_skill = pd.concat(
    [
        skill_table(subset, baseline="none", metric="WQL").assign(windows=name)
        for name, subset in subsets.items()
    ],
    ignore_index=True,
)
holiday_skill = holiday_skill[holiday_skill["model"] != "none"]
print("Saved", save_results(holiday_skill, "05_holiday_windows", PATHS).name)
for name, subset in subsets.items():
    print(f"\n{name} ({subset['window'].nunique()} windows), WQL vs no covariates:")
    rows = holiday_skill[holiday_skill["windows"] == name].set_index("model")
    for model in (HOLIDAY_ARM, "+calendar"):
        row = rows.loc[model]
        print(
            f"  {model:<20} {row['skill']:+7.1%}  [{row['ci_low']:+.1%}, {row['ci_high']:+.1%}]  "
            f"win {row['win_rate']:.0%}"
        )

## 6. What Chronos-2 will not take

One limit is worth stating plainly, because it bounds everything above: Chronos-2 accepts
**numeric and categorical covariates only**. There is no text input. A promotion's description,
an incident report, a weather warning in words — none of it goes in, and the paper names
multimodal inputs as future work rather than a feature.

In practice that means the engineering job is turning what you know into numbers and categories:
a holiday becomes a flag, a tariff becomes a level, a published forecast becomes a column. The
ladder above is exactly that job, done once.

## Summary

In [ ]:
epf_skill = load_results("05_epf_de_skill", PATHS).set_index("model")
ladder_skill = load_results("05_price_ladder_skill", PATHS).set_index("model")

print("EPF-DE (fev-bench task, 20 published windows), WQL vs the univariate task:")
for model in ("past_only", "known_future"):
    row = epf_skill.loc[model]
    print(
        f"  {model:<14} {row['skill']:+7.1%}  [{row['ci_low']:+.1%}, {row['ci_high']:+.1%}]  "
        f"win {row['win_rate']:.0%}"
    )

print("\nSpanish day-ahead price, WQL vs no covariates:")
for model in ladder_skill.index:
    row = ladder_skill.loc[model]
    print(
        f"  {model:<24} {row['skill']:+7.1%}  [{row['ci_low']:+.1%}, {row['ci_high']:+.1%}]  "
        f"win {row['win_rate']:.0%}"
    )

**Next:** [`06_explainability`](06_explainability.ipynb) asks which of these covariates the
forecast is actually using, with techniques that do not need the model to cooperate.